In [50]:
from websocket import create_connection
import json
import pandas as pd
import re
import datetime as dt
import textwrap

In [52]:
socket = 'wss://widgetdata.tradingview.com/socket.io/websocket'

#candlestick data
msg0 = '~m~55~m~{"m":"chart_create_session","p":["cs_a75tuXOstE8C",""]}'
msg1 = '~m~140~m~{"m":"resolve_symbol","p":["cs_a75tuXOstE8C","sds_sym_1","={\"adjustment\":\"splits\",\"session\":\"regular\",\"symbol\":\"NASDAQ:AAPL\"}"]}'
msg2 = '~m~82~m~{"m":"create_series","p":["cs_a75tuXOstE8C","sds_1","s1","sds_sym_1","15",300,""]}'

#quote data
msg3 = '~m~83~m~{"m":"quote_create_session","p":["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0"]}'

messages = []
ws = create_connection(socket)

def create_msg(ws, fun, arg):
    ms = json.dumps({"m": fun,"p": arg})
    msg = '~m~' + str(len(ms)) + '~m~' + ms
    ws.send(msg)



create_msg(ws, 'chart_create_session', ["cs_a75tuXOstE8C",""])
#initialiseert de websocket verbinding en geeft alle info die nodig is om data op te vragen

#create_msg(ws, 'quote_create_session', ["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0"])

create_msg(ws, 'resolve_symbol', ["cs_a75tuXOstE8C","sds_sym_1","={\"adjustment\":\"splits\",\"session\":\"regular\",\"symbol\":\"NASDAQ:AAPL\"}"])
#metadata waarvan de bulk van de data session holidays zijn oftwel dagen waarop de beurs gesloten zal zijn en selecteerd in ticker symbool

create_msg(ws, 'create_series', ["cs_a75tuXOstE8C","sds_1","s1","sds_sym_1","1",1,""])
#10 1 minute candles met timestamp, open, high, low, close, volume

while True:
    res = ws.recv()
    messages.append(res)
    print(res)
    if 'series_completed' in res:
        break

~m~293~m~{"session_id":"0.27229.1036_lon1-charts-wgt-1-tvbs-yw5ld-3","timestamp":1760109787,"timestampMs":1760109787866,"release":"release_208-76","studies_metadata_hash":"44aae310c18bc00c093e2de428dbee1bd4796e33","auth_scheme_vsn":2,"protocol":"json","via":"93.123.102.191:443","javastudies":["3.66"]}
~m~95~m~{"m":"series_loading","p":["cs_a75tuXOstE8C","sds_1","s1"],"t":1760109787,"t_ms":1760109787941}~m~4726~m~{"m":"symbol_resolved","p":["cs_a75tuXOstE8C","sds_sym_1",{"source2":{"country":"US","description":"Cboe One","exchange-type":"exchange","id":"BATS","name":"Cboe One","url":"https://markets.cboe.com/us/equities/overview/"},"currency_code":"USD","source_id":"BATS","session_holidays":"20000117,20000221,20000421,20000529,20000704,20000904,20001123,20001225,20010101,20010115,20010219,20010413,20010528,20010704,20010903,20011122,20011225,20020101,20020121,20020218,20020329,20020527,20020704,20020902,20021128,20021225,20030101,20030120,20030217,20030418,20030526,20030704,20030901,200

In [53]:
print(res)
with open('tradingview_data.json', 'w') as f:
    # Converteer naar compacte JSON string
    json_str = json.dumps(messages)
    
    # Wrap tekst naar maximale lijn lengte
    wrapped_text = textwrap.fill(json_str, width=120, break_long_words=True)
    f.write(wrapped_text)

~m~95~m~{"m":"series_loading","p":["cs_a75tuXOstE8C","sds_1","s1"],"t":1760109787,"t_ms":1760109787941}~m~4726~m~{"m":"symbol_resolved","p":["cs_a75tuXOstE8C","sds_sym_1",{"source2":{"country":"US","description":"Cboe One","exchange-type":"exchange","id":"BATS","name":"Cboe One","url":"https://markets.cboe.com/us/equities/overview/"},"currency_code":"USD","source_id":"BATS","session_holidays":"20000117,20000221,20000421,20000529,20000704,20000904,20001123,20001225,20010101,20010115,20010219,20010413,20010528,20010704,20010903,20011122,20011225,20020101,20020121,20020218,20020329,20020527,20020704,20020902,20021128,20021225,20030101,20030120,20030217,20030418,20030526,20030704,20030901,20031127,20031225,20040101,20040119,20040216,20040409,20040531,20040611,20040705,20040906,20041125,20041224,20050117,20050221,20050325,20050530,20050704,20050905,20051124,20051226,20060102,20060116,20060220,20060414,20060529,20060704,20060904,20061123,20061225,20070101,20070102,20070115,20070219,20070406,

In [54]:
patroon = re.compile(r'"s":(.*?),"ns"')
gevonden = re.search(patroon, res)
lijst_gevonden = json.loads(gevonden.group(1))

if gevonden:
    print(lijst_gevonden)
else:
    print("Geen match gevonden")

[{'i': 0, 'v': [1760109780.0, 249.69, 249.71, 249.57, 249.57, 4891.0]}]


In [55]:
data_for_df = []
for line in lijst_gevonden:
    print(line['v'])

    v_data = line['v']
    row = {
        'timestamp': dt.datetime.fromtimestamp(v_data[0]),
        'open' : v_data[1],
        'high' : v_data[2],
        'low' : v_data[3],
        'close' : v_data[4],
        'volume' : v_data[5]
    }
    data_for_df.append(row)

df = pd.DataFrame(data_for_df)
df

[1760109780.0, 249.69, 249.71, 249.57, 249.57, 4891.0]


,timestamp,open,high,low,close,volume
0,2025-10-10 17:23:00,249.69,249.71,249.57,249.57,4891.0
